In [14]:
import pandas as pd
import numpy as np
import joblib
from tqdm import tqdm
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

ind_hosp = pd.read_parquet('ind_hosp.parquet')
calibrated = joblib.load('lightgbm.pkl')

cols_to_drop = [ 
    'dischtime', 
    'current_date',
    'target_readmission_30d',
    'los'
]
cols_to_drop = [c for c in cols_to_drop if c in ind_hosp.columns]

X = ind_hosp.drop(columns=cols_to_drop)
bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)

patient_target = ind_hosp.groupby('subject_id')['target_readmission_30d'].max().reset_index()
patient_target.columns = ['subject_id', 'has_readmission']
train_val_ids, test_ids = train_test_split(
    patient_target['subject_id'],
    test_size=0.2,
    random_state=42,
    stratify=patient_target['has_readmission']
)

# sample_size = 1000
# test_patients_sample, _ = train_test_split(
#     test_ids,
#     train_size=sample_size,
#     random_state=42,
#     stratify=patient_target[patient_target['subject_id'].isin(test_ids)]['has_readmission']
# )
# test_patients = test_patients_sample.tolist()
test_patients = test_ids.tolist()

print(f"Test patients: {len(test_patients)}")

Test patients: 35915


In [15]:
icd_cols = [col for col in X.columns if col.startswith('icd_')]
ccsr_cols = [col for col in X.columns if col.startswith('ccsr_')]
lab_cols = [col for col in X.columns if col.startswith('lab_') and col.endswith('_daily')]

features_to_analyze = (
    icd_cols +
    ccsr_cols +
    lab_cols +
    [
        'num_diagnoses',
        'num_chronic',
        'comorbidity_score',
        'num_medications_daily',
        'prior_admissions_12mo',
        'cumulative_procedures',
        'cumulative_medications',
        'num_procedures_daily',
        'gender_male',
        'age'
    ]
)

features_to_analyze = [f for f in features_to_analyze if f in X.columns]
print(f"Analyzing {len(features_to_analyze)} features")

Analyzing 60 features


In [16]:
clinical_ranges = {
    'lab_50983_daily': {'low': 135, 'high': 147, 'norm': 141},
    'lab_50971_daily': {'low': 3.3, 'high': 5.1, 'norm': 4.2},
    'lab_50902_daily': {'low': 96, 'high': 108, 'norm': 102},
    'lab_50882_daily': {'low': 22, 'high': 32, 'norm': 27},
    'lab_50912_daily': {'low': 0.5, 'high': 1.2, 'norm': 0.85},
    'lab_51006_daily': {'low': 6, 'high': 20, 'norm': 13},
    'lab_50931_daily': {'low': 70, 'high': 100, 'norm': 85},
    'lab_50893_daily': {'low': 8.4, 'high': 10.3, 'norm': 9.35},
    'lab_50868_daily': {'low': 10, 'high': 18, 'norm': 14},
    'lab_51222_daily': {'low': 13.7, 'high': 17.5, 'norm': 15.6},
    'lab_51301_daily': {'low': 4, 'high': 10, 'norm': 7},
    'lab_51265_daily': {'low': 150, 'high': 400, 'norm': 275},
    'lab_51221_daily': {'low': 40, 'high': 51, 'norm': 45.5},
    'lab_51250_daily': {'low': 82, 'high': 98, 'norm': 90},
    'lab_51277_daily': {'low': 10.5, 'high': 15.5, 'norm': 13},
    'lab_50960_daily': {'low': 1.6, 'high': 2.6, 'norm': 2.1},
    'lab_50970_daily': {'low': 2.7, 'high': 4.5, 'norm': 3.6},
    'lab_51248_daily': {'low': 26, 'high': 32, 'norm': 29},
    'lab_51249_daily': {'low': 32, 'high': 37, 'norm': 34.5},
    'lab_51279_daily': {'low': 4.6, 'high': 6.1, 'norm': 5.35}
}

age_groups = {
    'young': (18, 40),
    'middle': (40, 65),
    'elderly': (65, 80),
    'very_elderly': (80, 120)
}

In [17]:
def get_age_norm(age):
    if age < 40: return 30
    elif age < 65: return 50
    elif age < 80: return 70
    else: return 85

def get_counterfactual_value(feature, patient_value, X):
    if feature.startswith('lab_'):
        if feature in clinical_ranges:
            ranges = clinical_ranges[feature]
            low = ranges['low']
            high = ranges['high']
            norm = ranges['norm']
            
            if low <= patient_value <= high:
                direction = 'normal'
            elif patient_value < low:
                direction = 'low'
            else:
                direction = 'high'
            
            return norm, direction, patient_value - norm
    
    if feature == 'age':
        age_norm = get_age_norm(patient_value)
        if patient_value < age_norm:
            return age_norm, 'younger', patient_value - age_norm
        else:
            return age_norm, 'older', patient_value - age_norm
    
    if feature == 'gender_male':
        return 1 if patient_value == 0 else 0, 'opposite', 0
    
    if feature in ['num_medications_daily', 'num_procedures_daily']:
        if patient_value > 0:
            return 0, 'remove', -patient_value
        else:
            return 1, 'add', 1
    
    if feature in ['cumulative_medications', 'cumulative_procedures']:
        mean_val = X[feature].mean()
        if patient_value > mean_val:
            return mean_val, 'reduce_to_mean', patient_value - mean_val
        else:
            return mean_val, 'increase_to_mean', patient_value - mean_val
    
    if feature in ['prior_admissions_12mo', 'comorbidity_score']:
        if patient_value > 0:
            return 0, 'remove', -patient_value
        else:
            return 1, 'add', 1
    
    if feature.startswith('icd_') or feature.startswith('ccsr_'):
        if patient_value == 1:
            return 0, 'remove', -1
        else:
            return 1, 'add', 1
    
    if feature in ['num_diagnoses', 'num_chronic']:
        if patient_value > X[feature].mean():
            return X[feature].mean(), 'reduce_to_mean', patient_value - X[feature].mean()
        else:
            return X[feature].mean(), 'increase_to_mean', patient_value - X[feature].mean()
    
    return X[feature].mean(), 'to_mean', 0

def predict_risk(model, data):
    p_t = model.predict_proba(data)[:, 1]
    risk = 1 - np.prod(1 - p_t)
    return risk

In [18]:
all_results = []
for patient_id in tqdm(test_patients, desc="Patients"):
    patient_data = X[X['subject_id'] == patient_id].copy()
    
    for hadm_id in patient_data['hadm_id'].unique():
        hadm_data = patient_data[patient_data['hadm_id'] == hadm_id].copy()
        hadm_data_no_ids = hadm_data.drop(['subject_id', 'hadm_id'], axis=1)
        
        risk_actual = predict_risk(calibrated, hadm_data_no_ids)
        n_days = len(hadm_data_no_ids)
        
        for feature in features_to_analyze:
            if feature not in hadm_data_no_ids.columns:
                continue
            
            hadm_counter = hadm_data_no_ids.copy()
            feature_col_idx = hadm_counter.columns.get_loc(feature)
            
            values = hadm_data_no_ids[feature].values
            counter_values = []
            directions = []
            deviations = []
            
            for day_idx, current_value in enumerate(values):
                counter_value, direction, deviation = get_counterfactual_value(
                    feature, current_value, X
                )
                hadm_counter.iloc[day_idx, feature_col_idx] = counter_value
                counter_values.append(counter_value)
                directions.append(direction)
                deviations.append(deviation)
            
            risk_counter = predict_risk(calibrated, hadm_counter)
            delta = risk_actual - risk_counter
            
            all_results.append({
                'subject_id': patient_id,
                'hadm_id': hadm_id,
                'feature': feature,
                'delta': delta,
                'risk_actual': risk_actual,
                'risk_counterfactual': risk_counter,
                'n_days': n_days,
                'mean_current_value': np.mean(values),
                'std_current_value': np.std(values),
                'min_current_value': np.min(values),
                'max_current_value': np.max(values),
                'mean_counterfactual': np.mean(counter_values),
                'mean_deviation': np.mean(deviations),
                'mode_direction': max(set(directions), key=directions.count)
            })

results_df = pd.DataFrame(all_results)

Patients: 100%|██████████| 35915/35915 [2:21:02<00:00,  4.24it/s]  


In [19]:
deltas_by_hadm = results_df

delta_stats = deltas_by_hadm.groupby('feature', as_index=False).agg({
    'delta': ['mean', 'std', 'min', 'max', 'count'],
    'risk_actual': 'mean',
    'risk_counterfactual': 'mean',
    'n_days': 'mean',
    'mean_current_value': 'mean',
    'std_current_value': 'mean',
    'min_current_value': 'mean',
    'max_current_value': 'mean',
    'mean_counterfactual': 'mean',
    'mean_deviation': 'mean',
    'mode_direction': lambda x: x.mode()[0] if len(x) > 0 else 'unknown'
}).round(4)

delta_stats.columns = [
    'feature',
    'delta_mean', 'delta_std', 'delta_min', 'delta_max', 'n_hadm',
    'risk_mean', 'risk_counterfactual_mean', 'n_days_mean',
    'mean_current_value_mean', 'std_current_value_mean',
    'min_current_value_mean', 'max_current_value_mean',
    'mean_counterfactual_mean', 'mean_deviation_mean',
    'most_common_direction'
]

delta_abs_by_hadm = deltas_by_hadm.groupby('feature')['delta'].apply(
    lambda x: x.abs().mean()
).reset_index()
delta_abs_by_hadm.columns = ['feature', 'delta_abs_mean']

delta_stats = delta_stats.merge(delta_abs_by_hadm, on='feature', how='left')
delta_stats = delta_stats.sort_values('delta_mean', ascending=False)

delta_stats.to_csv('delta_summary.csv', index=False)
results_df.to_csv('total_deltas.csv', index=False)

In [20]:
print("\nFactors that increase risk (Δ > 0):")
positive = delta_stats[delta_stats['delta_mean'] > 0].sort_values('delta_mean', ascending=False).head(10)
for _, row in positive.iterrows():
    print(f"  {row['feature']}: Δ = {row['delta_mean']:+.4f} ± {row['delta_std']:.4f} [{row['most_common_direction']}]")

print("\nFactors that decrease risk (Δ < 0):")
negative = delta_stats[delta_stats['delta_mean'] < 0].sort_values('delta_mean', ascending=True).head(10)
for _, row in negative.iterrows():
    print(f"  {row['feature']}: Δ = {row['delta_mean']:+.4f} ± {row['delta_std']:.4f} [{row['most_common_direction']}]")


Factors that increase risk (Δ > 0):
  prior_admissions_12mo: Δ = +0.0292 ± 0.1253 [add]
  comorbidity_score: Δ = +0.0235 ± 0.0497 [remove]
  lab_51277_daily: Δ = +0.0149 ± 0.0279 [normal]
  lab_51279_daily: Δ = +0.0108 ± 0.0171 [low]
  ccsr_CIR019: Δ = +0.0102 ± 0.0194 [add]
  lab_51221_daily: Δ = +0.0097 ± 0.0135 [low]
  ccsr_EXT027: Δ = +0.0083 ± 0.0186 [add]
  ccsr_END009: Δ = +0.0065 ± 0.0121 [add]
  lab_50983_daily: Δ = +0.0042 ± 0.0123 [normal]
  ccsr_END003: Δ = +0.0040 ± 0.0098 [add]

Factors that decrease risk (Δ < 0):
  num_diagnoses: Δ = -0.0132 ± 0.0189 [increase_to_mean]
  num_chronic: Δ = -0.0119 ± 0.0186 [increase_to_mean]
  num_medications_daily: Δ = -0.0116 ± 0.0299 [remove]
  ccsr_BLD003: Δ = -0.0105 ± 0.0163 [add]
  ccsr_END011: Δ = -0.0059 ± 0.0117 [add]
  ccsr_CIR007: Δ = -0.0053 ± 0.0160 [add]
  cumulative_procedures: Δ = -0.0047 ± 0.0195 [increase_to_mean]
  age: Δ = -0.0035 ± 0.0125 [older]
  icd_F329: Δ = -0.0034 ± 0.0113 [add]
  lab_50971_daily: Δ = -0.0032 ±